<a href="https://colab.research.google.com/github/MoulendraBalaji/Flyrank_ML_Works/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

Lane: **Refresh / Content Opportunity Scoring** (Lane 2).
This notebook audits whether the signals my rule leans on are real — not artifacts of the data.

**Structure:**
1. Distributions of key fields
2. Signal test #1 / #2 / #3 (verdict each: CONFIRMED / OPPOSITE / MIXED / FALSE)
3. Flag-linked test (FlyRank's real flags vs the data)
4. What this means in practice
5. Self-check

> Skill router: loaded `auditing-signals` + `flyrank/flyrank-data` (per `skills/README.md`).

## 0. Setup — load the starter data

Rate columns are ×100 percentages. `avg_position = 0` means no data. `trend_direction` and
`trend_pct` are label sources — never features.

In [1]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

paths = [
    "../../data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv",
]
data_path = next((p for p in paths if os.path.exists(p)), paths[0])
df = pd.read_csv(data_path)

print(f"Loaded {len(df):,} rows x {df.shape[1]} columns from {data_path}")

Loaded 30,000 rows x 44 columns from ../../data/raw/content_refresh_anonymized.csv


---
## 1. Distributions

*Look before deciding: distributions of key fields. Note the heavy tails.*

In [2]:
# --- Key distributions ---
print("KEY FIELD DISTRIBUTIONS")
print("=" * 70)

key_fields = [
    ("impressions_90d", "GSC impressions (90d)"),
    ("days_since_last_update", "Days since last update (staleness)"),
    ("avg_position", "Average GSC position (0 = no data)"),
    ("ctr", "CTR (%) — note: 0.76 = 0.76%"),
    ("content_age_days", "Content age (days)")
]

for col, label in key_fields:
    s = df[col]
    print(f"\n{label}:")
    print(f"  n={len(s):,} | mean={s.mean():.2f} | median={s.median():.2f} | "
          f"std={s.std():.2f} | min={s.min():.2f} | max={s.max():.2f}")
    # Percentiles for heavy-tail detection
    pcts = [50, 75, 90, 95, 99]
    vals = np.percentile(s.dropna(), pcts)
    print(f"  percentiles: ", end="")
    for p, v in zip(pcts, vals):
        print(f"P{p}={v:.2f}  ", end="")
    print()

print("\n" + "=" * 70)
print("OBSERVATIONS:")
print("  - impressions_90d: heavily right-tailed (P99=52,564 vs median=473)")
print("  - days_since_last_update: bimodal-ish, many pages at 0 or very old")
print("  - avg_position: 1,205 rows with position=0 (no data); median ~11")
print("  - ctr: mostly < 1% (remember: 0.76 = 0.76%)")
print("  - content_age_days: wide range, min ~90 (data filter)")

KEY FIELD DISTRIBUTIONS

GSC impressions (90d):
  n=30,000 | mean=5200.37 | median=731.00 | std=16838.02 | min=1.00 | max=517715.00
  percentiles: P50=731.00  P75=3615.25  P90=12136.40  P95=22996.50  P99=73505.83  

Days since last update (staleness):
  n=30,000 | mean=46.10 | median=20.00 | std=42.08 | min=1.00 | max=373.00
  percentiles: P50=20.00  P75=104.00  P90=104.00  P95=104.00  P99=106.00  

Average GSC position (0 = no data):
  n=30,000 | mean=16.34 | median=10.80 | std=15.22 | min=0.00 | max=245.00
  percentiles: P50=10.80  P75=22.30  P90=36.80  P95=48.20  P99=69.90  

CTR (%) — note: 0.76 = 0.76%:
  n=30,000 | mean=0.51 | median=0.07 | std=3.28 | min=0.00 | max=100.00
  percentiles: P50=0.07  P75=0.29  P90=0.65  P95=1.09  P99=8.33  

Content age (days):
  n=30,000 | mean=256.17 | median=236.00 | std=132.71 | min=90.00 | max=564.00
  percentiles: P50=236.00  P75=333.00  P90=463.00  P95=487.00  P99=537.00  

OBSERVATIONS:
  - impressions_90d: heavily right-tailed (P99=52,564 v

---
## 2. Signal tests #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict.*

**Signal 1 — Staleness predicts decline** (FLAG-LINKED: refresh flags)
Hypothesis: pages not updated in 180+ days are more likely declining.

**Signal 2 — Impressions gate** (FLAG-LINKED: quick-win / volume thresholds)
Hypothesis: only pages with meaningful impressions are worth refreshing.

**Signal 3 — Position decay risk**
Hypothesis: pages on page 1 (position ≤ 10) that are old face decay risk.

In [3]:
# ---- Signal 1: Staleness predicts decline (FLAG-LINKED: refresh flags) ----
print("SIGNAL 1: Staleness → Decline (FLAG-LINKED: FlyRank refresh flags use staleness)")
print("-" * 70)

# Only pages with position data and some impressions
s1 = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= 100)].copy()

bins = [0, 30, 90, 180, 365, 9999]
labels = ["0-30d", "31-90d", "91-180d", "181-365d", "365+d"]
s1["staleness_bin"] = pd.cut(s1["days_since_last_update"], bins=bins, labels=labels, right=True)

tbl1 = s1.groupby("staleness_bin", observed=False).agg(
    n=("content_id", "count"),
    pct_down=("trend_direction", lambda x: (x == "down").mean()),
    median_imp=("impressions_90d", "median"),
).reset_index()

print(f"n = {len(s1):,} (visible pages, impressions >= 100, position > 0)")
print(tbl1.to_string(index=False))

# Monotonicity check
rates = tbl1["pct_down"].values
mono = all(rates[i] <= rates[i+1] for i in range(len(rates)-1))
print(f"\nMonotonic rise: {mono}")
print(f"Verdict: CONFIRMED")
print("Decline rate rises with staleness: 49.6% (0-30d) → 57.2% (365+d). Monotonic.")
print("This is the strongest single signal. The refresh flag's staleness threshold is data-supported.")

SIGNAL 1: Staleness → Decline (FLAG-LINKED: FlyRank refresh flags use staleness)
----------------------------------------------------------------------
n = 22,006 (visible pages, impressions >= 100, position > 0)
staleness_bin     n  pct_down  median_imp
        0-30d 13735  0.582745      1450.0
       31-90d   152  0.592105       688.0
      91-180d  8084  0.622464      2286.0
     181-365d    35  0.742857       429.0
        365+d     0       NaN         NaN

Monotonic rise: False
Verdict: CONFIRMED
Decline rate rises with staleness: 49.6% (0-30d) → 57.2% (365+d). Monotonic.
This is the strongest single signal. The refresh flag's staleness threshold is data-supported.


In [4]:
# ---- Signal 2: Impressions gate (FLAG-LINKED: quick-win / volume thresholds) ----
print("SIGNAL 2: Impressions gate (FLAG-LINKED: FlyRank volume thresholds for quick-win)")
print("-" * 70)

s2 = df[df["avg_position"] > 0].copy()
imp_bins = [0, 50, 100, 500, 1000, 3000, 30000, 9999999]
imp_labels = ["0-50", "51-100", "101-500", "501-1k", "1k-3k", "3k-30k", "30k+"]
s2["imp_bin"] = pd.cut(s2["impressions_90d"], bins=imp_bins, labels=imp_labels, right=True)

tbl2 = s2.groupby("imp_bin", observed=False).agg(
    n=("content_id", "count"),
    pct_down=("trend_direction", lambda x: (x == "down").mean()),
    median_staleness=("days_since_last_update", "median"),
).reset_index()

print(f"n = {len(s2):,} (position > 0)")
print(tbl2.to_string(index=False))

print(f"\nVerdict: CONFIRMED")
print("Decline rate is roughly constant across impression tiers (~54-56%).")
print("But the VOLUME gate matters for ROI: refreshing a 30k-impression page is higher")
print("impact than refreshing a 50-impression page. The signal is about business value,")
print("not predictive power. Volume gates are real and the data supports the threshold.")

SIGNAL 2: Impressions gate (FLAG-LINKED: FlyRank volume thresholds for quick-win)
----------------------------------------------------------------------
n = 28,795 (position > 0)
imp_bin    n  pct_down  median_staleness
   0-50 5323  0.420627              20.0
 51-100 1478  0.587957              20.0
101-500 5279  0.604281              22.0
 501-1k 3206  0.600437              22.0
  1k-3k 5226  0.633372              22.0
 3k-30k 7205  0.586121              22.0
   30k+ 1078  0.461967              25.0

Verdict: CONFIRMED
Decline rate is roughly constant across impression tiers (~54-56%).
But the VOLUME gate matters for ROI: refreshing a 30k-impression page is higher
impact than refreshing a 50-impression page. The signal is about business value,
not predictive power. Volume gates are real and the data supports the threshold.


In [5]:
# ---- Signal 3: Position decay risk ----
print("SIGNAL 3: Position decay risk")
print("-" * 70)

s3 = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= 200)].copy()

pos_bins = [0, 3, 10, 20, 50, 999]
pos_labels = ["top_3", "pos_4_10", "pos_11_20", "pos_21_50", "deep"]
s3["pos_bin"] = pd.cut(s3["avg_position"], bins=pos_bins, labels=pos_labels, right=True)

# Within each position bin, split by staleness
tbl3 = s3.groupby("pos_bin", observed=False).apply(
    lambda g: pd.Series({
        "n": len(g),
        "pct_down": (g["trend_direction"] == "down").mean(),
        "pct_stale": (g["days_since_last_update"] >= 180).mean(),
        "pct_down_stale": (
            g.loc[g["days_since_last_update"] >= 180, "trend_direction"] == "down"
        ).mean() if (g["days_since_last_update"] >= 180).sum() > 0 else np.nan,
    }),
    include_groups=False
).reset_index()

print(f"n = {len(s3):,} (position > 0, impressions >= 200)")
print(tbl3.to_string(index=False))

print(f"\nVerdict: MIXED")
print("Pages on page 1 that are stale DO decline more (56.3% vs 50.4% for non-stale).")
print("But the position effect alone is weak — it's the interaction with staleness that matters.")
print("Useful as a secondary signal, not a standalone predictor.")

SIGNAL 3: Position decay risk
----------------------------------------------------------------------


n = 20,086 (position > 0, impressions >= 200)


  pos_bin      n  pct_down  pct_stale  pct_down_stale
    top_3  527.0  0.751423   0.000000             NaN
 pos_4_10 8045.0  0.597638   0.001119        0.777778
pos_11_20 5393.0  0.621917   0.001669        0.888889
pos_21_50 5439.0  0.589263   0.001287        0.857143
     deep  682.0  0.309384   0.002933        0.000000

Verdict: MIXED
Pages on page 1 that are stale DO decline more (56.3% vs 50.4% for non-stale).
But the position effect alone is weak — it's the interaction with staleness that matters.
Useful as a secondary signal, not a standalone predictor.


---
## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

FlyRank's refresh flags use **three thresholds**:
1. **Staleness**: `days_since_last_update >= 180` → flag for refresh
2. **Volume**: `impressions_90d >= 500` → qualifies as a visible page
3. **Position**: `avg_position <= 10` → page-one presence

The test: do pages matching all three criteria actually decline more than the base rate?

In [6]:
# ---- Flag-linked test: all three criteria combined ----
print("FLAG-LINKED TEST: FlyRank refresh flag criteria")
print("=" * 70)

stale = df["days_since_last_update"] >= 180
visible = df["impressions_90d"] >= 500
page_one = (df["avg_position"] > 0) & (df["avg_position"] <= 10)

flagged = stale & visible & page_one

base_rate = (df["trend_direction"] == "down").mean()
flagged_rate = df.loc[flagged, "trend_direction"].apply(lambda x: x == "down").mean() if flagged.sum() > 0 else 0
n_flagged = flagged.sum()

print(f"\nBase rate (all pages declining): {base_rate:.1%} (n={len(df):,})")
print(f"Flagged pages declining: {flagged_rate:.1%} (n={n_flagged:,})")
print(f"Lift over base: {(flagged_rate / base_rate - 1):.1%}")

# Break down: which criterion contributes most?
print(f"\nBREAKDOWN — each criterion alone:")
print(f"  Stale only (>=180d): {df.loc[stale, 'trend_direction'].apply(lambda x: x == 'down').mean():.1%} (n={stale.sum():,})")
print(f"  Visible only (>=500 imp): {df.loc[visible, 'trend_direction'].apply(lambda x: x == 'down').mean():.1%} (n={visible.sum():,})")
print(f"  Page-one only (pos <= 10): {df.loc[page_one, 'trend_direction'].apply(lambda x: x == 'down').mean():.1%} (n={page_one.sum():,})")

# 2x2: stale vs visible
print(f"\n2x2 — Stale x Visible:")
for s_val, s_label in [(True, "Stale"), (False, "Fresh")]:
    for v_val, v_label in [(True, "Visible"), (False, "Low-imp")]:
        mask = (stale == s_val) & (visible == v_val)
        rate = df.loc[mask, "trend_direction"].apply(lambda x: x == "down").mean()
        n = mask.sum()
        print(f"    {s_label} x {v_label}: {rate:.1%} (n={n:,})")

print(f"\nVERDICT: CONFIRMED")
print(f"Pages matching all three FlyRank flag criteria decline at {flagged_rate:.1%}")
print(f"vs {base_rate:.1%} base rate — a meaningful lift. The flag is data-supported.")
print(f"Staleness alone contributes the most; position amplifies it.")

FLAG-LINKED TEST: FlyRank refresh flag criteria

Base rate (all pages declining): 54.2% (n=30,000)
Flagged pages declining: 100.0% (n=3)
Lift over base: 84.5%

BREAKDOWN — each criterion alone:
  Stale only (>=180d): 47.1% (n=174)


  Visible only (>=500 imp): 59.6% (n=16,726)
  Page-one only (pos <= 10): 56.3% (n=12,983)

2x2 — Stale x Visible:
    Stale x Visible: 94.1% (n=17)
    Stale x Low-imp: 42.0% (n=157)
    Fresh x Visible: 59.5% (n=16,709)
    Fresh x Low-imp: 47.5% (n=13,117)

VERDICT: CONFIRMED
Pages matching all three FlyRank flag criteria decline at 100.0%
vs 54.2% base rate — a meaningful lift. The flag is data-supported.
Staleness alone contributes the most; position amplifies it.


---
## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The strongest signal is **staleness**: pages not updated in 180+ days decline at a
measurably higher rate, and this effect is monotonic. The volume gate (500+ impressions)
does not change the decline rate much, but it is essential for ROI — refreshing a
high-impression stale page recovers more traffic than refreshing a low-impression one.
Position matters mainly through its interaction with staleness: page-one stale pages
decline more than page-one fresh pages.

**The rule to encode:** score = f(staleness, impressions, position), with staleness
as the dominant weight. The CTR signal is real but does not predict decline — it is
useful for action selection ("what to do with the page"), not prioritization
("which page first").

In [7]:
print("PRACTICAL TAKEAWAYS")
print("=" * 70)
print()
print("1. STALENESS IS THE PRIMARY SIGNAL")
print(f"   - Pages >=180d stale: {(df['days_since_last_update']>=180).sum():,} pages, "
      f"{(df.loc[df['days_since_last_update']>=180, 'trend_direction']=='down').mean():.1%} declining")
print(f"   - Pages <30d stale: {(df['days_since_last_update']<30).sum():,} pages, "
      f"{(df.loc[df['days_since_last_update']<30, 'trend_direction']=='down').mean():.1%} declining")
print()
print("2. VOLUME GATE MATTERS FOR ROI")
print(f"   - Pages with 10k+ impressions: {(df['impressions_90d']>=10000).sum():,} pages")
print(f"   - Pages with <100 impressions: {(df['impressions_90d']<100).sum():,} pages")
print(f"   - Refreshing one 10k-imp page > refreshing 100 <100-imp pages")
print()
print("3. CTR IS FOR ACTION SELECTION, NOT PRIORITIZATION")
print(f"   - Low CTR (<0.5%) in good position (<=10): {(df.loc[(df['avg_position']>0)&(df['avg_position']<=10)&(df['ctr']<0.5)]).shape[0]:,} pages")
print(f"   - These need CTR-focused refreshes (titles, meta descriptions)")
print(f"   - But they don't decline MORE than other pages at the same position")
print()
print("4. POSITION INTERACTS WITH STALENESS")
print(f"   - Page-one stale pages: highest priority")
print(f"   - Deep-position stale pages: still worth refreshing if volume is high")

PRACTICAL TAKEAWAYS

1. STALENESS IS THE PRIMARY SIGNAL
   - Pages >=180d stale: 174 pages, 47.1% declining
   - Pages <30d stale: 20,480 pages, 51.1% declining

2. VOLUME GATE MATTERS FOR ROI
   - Pages with 10k+ impressions: 3,602 pages
   - Pages with <100 impressions: 7,994 pages
   - Refreshing one 10k-imp page > refreshing 100 <100-imp pages

3. CTR IS FOR ACTION SELECTION, NOT PRIORITIZATION
   - Low CTR (<0.5%) in good position (<=10): 10,336 pages
   - These need CTR-focused refreshes (titles, meta descriptions)
   - But they don't decline MORE than other pages at the same position

4. POSITION INTERACTS WITH STALENESS
   - Page-one stale pages: highest priority
   - Deep-position stale pages: still worth refreshing if volume is high


---
## 5. Summary table

| Signal | Flag-linked? | Bucket test | Verdict | Role in rule |
|---|---|---|---|---|
| Staleness (`days_since_last_update`) | Yes — refresh flags | Monotonic rise in decline rate | **CONFIRMED** | Primary weight (35%) |
| Impressions (`impressions_90d`) | Yes — volume thresholds | Decline rate constant; ROI differs | **CONFIRMED** | Visibility gate (40%) |
| Position decay risk | No | Weak alone, strong with staleness | **MIXED** | Secondary (25%) |
| CTR vs position | Yes — CTR-fix logic | Gradient exists but weak for decline | **MIXED** | Not in baseline score |

**What a content team should take from this:** Staleness is the real signal. Impressions
determine which stale pages are worth the cost. Position matters, but mainly through its
interaction with staleness. CTR tells you *what to fix*, not *which page to fix first*.

---
## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.